# PARAFAC ANALYSIS OF SINGLE-LEG HOP LANDING AFTER ACLR

#### Copyright (C) 2026 by the University of Melbourne

#### Authors: Runqi Zhao, with contributions from Prasanna Sritharan and Mario Andrés Munoz

# 1 - Importing Libraries
For this work, standard libraries such as pandas, numpy, sklearn and matplotlib will be used. Tensorly will be used to deal with Parafac.

In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
from time import time
import math
import pprint
import random

import tensorly as tl # tensor process library
from tensorly.decomposition import parafac2 # parafac2 tensor decomposition algorithm
from tensorly.decomposition import CP, parafac # parafac1 and cp decomposition algorithm
from sklearn.preprocessing import normalize # normalize tool

import scipy.io as sio # read matlab file
import seaborn # plot heat map

import pickle # file saver
import dataframe_image as dfi # dataframe saver


# Matplotlib: write text as text not path
from matplotlib import rc
rc("pdf", fonttype=42)
rc("font", **{'family':'sans-serif','sans-serif':['Arial']})

# 2 - Data collection and pre-processing
The dataset corresponds to the Hop forward landing tests, provided by the Centre for Health, Exercise, and Sports Medicine, University of Melbourne. The data is in the file 'aclr_hopfordist_landing_timehistories_all.csv' and was imported using the pandas package. Hop distance was normalised to 100% of leg length(greater trochanter to floor). It is noteworthy that a subject may have more than one trial associated to it. Besides, the muscle force variable is calculated with the weight of the subject and the moment variable is calculated with the height and the weight of the subject. Therefore, the data was pre-processed by the height and the weight of the subject. The angle was normalised considering the direction of movement.

In [ ]:
raw_data = pd.read_csv("aclr_hopfordist_landing_timehistories_all.csv")
ACLR = raw_data[raw_data.group=="ACLR"] # ACLR data
CTRL = raw_data[raw_data.group=="CTRL"] # CONTROL data

variables_labels = raw_data.variable.unique()
variables_angles = variables_labels[1:14] # angles data
variables_moments = variables_labels[14:24] # moments data
variables_force = variables_labels[[24,25,26, 43,44,45, 51, 52,53,54, 55,56 , 57, 30,31,32,33]] # force data

real_variables_force = ["glut_med", "glut_max", "rect_fem", "vasti", "gas", "soleus", "hamstring"]
real_variables = np.concatenate((variables_angles, variables_moments, real_variables_force)) # re-format variable labels

print(pd.Series(real_variables))

# process data by each row
# Input: unique row id, and trial id
# Output: Variable * Time (30*101)
def Select_variables_tims_by_id_trial(id, trial, data):
    weight = data[(data.id == id) & (data.trial == trial)].mass.iloc[0]
    height = data[(data.id == id) & (data.trial == trial)].height.iloc[0]

    angle_data = data[(data.id == id) & (data.trial == trial) & (data.variable.isin(variables_angles))].iloc[:, -101:]
    moment_data = data[(data.id == id) & (data.trial == trial) & (data.variable.isin(variables_moments))].iloc[:, -101:]
    moment_data = moment_data/(weight*height*10) # Normalize by weight and height
    # force data are made of glutmed, glutmax, rect_fem, vas, soleus, gas, and hamstring
    force_data = data[(data.id == id) & (data.trial == trial) & (data.variable.isin(variables_force))].iloc[:, -101:]
    force_data = force_data/(weight*10) # Normalize by weight
    glutmed = pd.DataFrame(force_data.iloc[0:3, :].sum(axis=0)).T
    glutmax = pd.DataFrame(force_data.iloc[3:6, :].sum(axis=0)).T
    rect_fem = pd.DataFrame(force_data.iloc[6,:]).T
    vas_lat = pd.DataFrame(force_data.iloc[7:10, :].sum(axis=0)).T
    gas = pd.DataFrame(force_data.iloc[10:12, :].sum(axis=0)).T
    soleus = pd.DataFrame(force_data.iloc[12,:]).T
    hamstring = pd.DataFrame(force_data.iloc[13:, :].sum(axis=0)).T

    variables = pd.concat([angle_data, moment_data, glutmed, glutmax, rect_fem, vas_lat, gas, soleus, hamstring], ignore_index=True)
    return variables.T.to_numpy()

# process each row
def process_data(data):
    result = []
    unique_ids = data.id.unique()
    for id in unique_ids:
        unique_trials = data[data.id == id].trial.unique()
        for trial in unique_trials:
            query = Select_variables_tims_by_id_trial(id, trial, data)
            result.append(query)
    return np.array(result)

# process ACLR data
processed_ACLR = process_data(ACLR)
np.random.seed(1)
# class balance
processed_ACLR = processed_ACLR[np.random.choice(processed_ACLR.shape[0], 151, replace=False)]
# process CTRL data
processed_CTRL = process_data(CTRL)

# data including aclr and control groups
tmp_tensor = np.concatenate((processed_ACLR, processed_CTRL), axis=0)
orig_data_tensor = tmp_tensor.copy()

## Rescale data
The data for different variables should fall in the same range. The angle and moment have positive value and negative value, refering to the direction of movement. The muscle variables are all positive values, the lowest value of muscle variable could potentially be zero, meaning that that part of body is not striking. After rescaling, the angle and moment is supposed to fall in range [-1, 1], the muscle is supposed to be between [0,1]

In [ ]:
# Rescale data
angle = tmp_tensor[:, : , 0:13]
moments = tmp_tensor[:, : , 13:23]
muscle = tmp_tensor[:, :, 23:]
# Max Rescale - preserve negatives and positives
for index in range(angle.shape[-1]):
    max_ = angle[:, :, index].max()
    min_ = angle[:, :, index].min()
    angle[:, :, index] = angle[:, :, index]/max(abs(max_), abs(min_))

for index in range(moments.shape[-1]):
    max_ = moments[:, :, index].max()
    min_ = moments[:, :, index].min()
    moments[:, :, index] = moments[:, :, index]/max(abs(max_), abs(min_))

for index in range(muscle.shape[-1]):
    max_ = muscle[:, :, index].max()
    min_ = muscle[:, :, index].min()
    muscle[:, :, index] = muscle[:, :, index]/max_

# final data for modeling
tensor = np.concatenate((angle, moments, muscle), axis=2)

# 3 - Select the proper number of component.
It is difficult to decide the best rank of a PARAFAC model. This area is not very well founded yet, and research is absolutely called for. Sometimes the increase in the residual variance is not very pronounced which makes it difficult to correctly estimate the proper rank of the model. This situation can be even worse in PARAFAC.
Elbow method is used here to determine the proper number of component used for parafac.

In [ ]:
start = time()
perfs = []
print("n\ttime")
for n in range(1, 31):
    
    # Perform PARAFAC
    fac = parafac(tensor, rank=n, normalize_factors=True, linesearch=True)
    
    # Calculate processing time
    split = time()
    splitdur = split - start
    perf = [n, splitdur]
    perfs.append(perf)
    print("%d\t%0.1f" % tuple(perf))
    
    # Pickle it
    with open('./SelectN/excel_fac_n={}.pkl'.format(n), 'wb') as outp:
        pickle.dump(fac, outp, pickle.HIGHEST_PROTOCOL)

end = time()
perfsdf = pd.DataFrame(perfs, columns=["n", "time"])

## VAF
The variance accounted for is the metric here to judge the performance of the model. The variance accounted for is calculated by the residual, which is the difference between the reconstruction tensor and the original data. The model with a bigger number of components is supposed to have a higher VAF.

In [ ]:
def vaf(rec, tensor):
    return 1 - tl.norm(rec-tensor)/tl.norm(tensor)

plot the ‘variance accounted for’ for each parafac model, and select the proper model where the VAF stops increasing rapidly.

In [ ]:
list_vaf = []
tvafs = []
for n in range(1, 31):
    with open('./SelectN/excel_fac_n={}.pkl'.format(n), 'rb') as input:
        new_fac = pickle.load(input)
        rec = tl.cp_to_tensor(new_fac)
        curvaf = vaf(rec, tensor)
        list_vaf.append(curvaf)
        tvafs.append([n, curvaf])

# Merge with running times
tvafsdf = pd.DataFrame(tvafs, columns=["n", "total_vaf"])
ptsdf = pd.merge(tvafsdf, perfsdf, on="n")
print(ptsdf)
ptsdf.to_csv("parafac_total_vaf_time.txt", sep="\t")

figure = plt.figure(dpi=300)
figure.patch.set_facecolor('white')
plt.title("Total VAF vs No. of components")
plt.xlabel("No. of components")
plt.ylabel("Variance accounted for")
plt.plot(list_vaf, 'r-')

#plt.savefig("fig2_total_VAF_vs_no_of_components.pdf", bbox_inches="tight")
#plt.savefig("fig2_total_VAF_vs_no_of_components.png", bbox_inches="tight")


After observing the data, We found out that when the number of component is equal to 11, where the VAF is around 80%, the prafac model best fit the our data.

In [ ]:
with open('./SelectN/excel_fac_n={}.pkl'.format(11), 'rb') as input:
    fac = pickle.load(input)
vaf(tl.cp_to_tensor(fac), tensor)

# 4 - Pre-process loading vectors of parafac
Loading vectors in parafac is given in format of (dimension, loading vector, component). As we want to analyze the performance of each individual component, the factors of parafac are preprocessed in proper format. The VAF of individual component might be noisy because the individual component may not have sufficient ability to recover the original data from its limited information, so we re-define the power of one component by the subduction of the VAF of all components and the VAF of the rest components. VAF(A) = VAF(ABC) - VAF(BC)

In [ ]:
def process_components(list_of_indexes, fac_sample):
    rank = len(list_of_indexes)
    dim1 = fac_sample.factors[0][:, list_of_indexes].reshape(302, rank)
    dim2 = fac_sample.factors[1][:, list_of_indexes].reshape(101, rank)
    dim3 = fac_sample.factors[2][:, list_of_indexes].reshape(30, rank)
    factor = [dim1, dim2, dim3]
    return factor

def process_components_by_rank(list_of_indexes, fac_sample):
    factors_by_rank = []
    for index in list_of_indexes:
        dim1 = fac_sample.factors[0][:,index].reshape(302,1)
        dim2 = fac_sample.factors[1][:,index].reshape(101,1)
        dim3 = fac_sample.factors[2][:,index].reshape(30,1)
        factor = [dim1, dim2, dim3]
        factors_by_rank.append(factor)

    return factors_by_rank

def compute_vaf_by_component_straight(list_of_indexes, fac_sample, tensor):
    vafs = []
    for index in list_of_indexes:
        reconstruct = tl.cp_to_tensor((fac_sample.weights[[index]], process_components([index], fac_sample)))
        vaf_ = vaf(reconstruct, tensor)
        vafs.append(vaf_)
    return vafs

def compute_vaf_by_component_greedy(original_rank, fac_sample, tensor):
    vaf_dict = {}
    pre_vaf = vaf(tl.cp_to_tensor(fac_sample), tensor)
    list_of_indexes = list(range(original_rank))
    result = []
    for cp in list_of_indexes:
        fac_indexes = list(range(original_rank))
        fac_indexes.remove(cp)
        reconstruct = tl.cp_to_tensor((fac_sample.weights[fac_indexes], process_components(fac_indexes, fac_sample)))
        result.append(pre_vaf - vaf(reconstruct, tensor))

    result = np.array(result)
    result = normalize([result], norm="l1")[0] * pre_vaf
    for index, value in enumerate(result):
        vaf_dict[index] = round(value, 3)

    return vaf_dict

result = compute_vaf_by_component_greedy(11, fac, tensor)

# 5 - Find the uncorrelated components
As the components computed by parafac are not orthogonal to each other, some components might be highly correlated. These components will make the interpretation of components complex, so the extraction of the most valuable factors out of all factors is critical. Here we exclude the component which contributes least to the model iteratively until the number of component is reduced to the amount we set.

In [ ]:
def greedy_permutation(r, k, fac_sample, tensor_sample):
    combinations = list(range(r))
    flag = True
    while flag:
        best_vaf_ = -10
        combinations_copy = combinations.copy()
        for index, value in enumerate(combinations_copy):
            res = combinations_copy[:index] + combinations_copy[index + 1:]
            reconstruct = tl.cp_to_tensor((fac_sample.weights[res], process_components(res, fac_sample)))
            vaf_ = vaf(reconstruct, tensor_sample)
            if vaf_ > best_vaf_:
                best_vaf_ = vaf_
                combinations = res
            if len(combinations) == k:
                flag = False
    return combinations

list_vaf_selected_cps = []
print("n\tTotal VAF")
for i in range(1, 11):
    t_rank = 11
    k_ = i
    optimal = greedy_permutation(t_rank, k_, fac, tensor)
    sum = 0
    for j in optimal:
        sum += result[j]
    print("%d\t%0.3f" % (i, sum))
    prevsum = sum
    list_vaf_selected_cps.append(sum)

figure = plt.figure(dpi=300)
figure.patch.set_facecolor('white')
plt.title("Total VAF vs No. of Components")
plt.xlabel("No. of selected components")
plt.ylabel("Variance accounted for")
plt.plot(np.linspace(1,10,10), list_vaf_selected_cps, 'r-')

#plt.savefig("fig3_total_VAF_after_BCE.pdf", bbox_inches="tight")
#plt.savefig("fig3_total_VAF_after_BCE.png", bbox_inches="tight")


By observing the graph, we see that when the number of component is 6, the total VAF of the model stops increasing rapidly, so we chose 6 components to be our optimal solution.

## Combine VAF plots for publication
Left: total VAF vs no. of components
Right: total VAF vs no. of components after BCE

In [ ]:
def VAF_combined_plots():

    # Generate plots
    fig, axs = plt.subplots(nrows=1, ncols=2, layout="constrained", figsize=(6, 3))

    # Total VAF vs no. of components
    axs[0].plot(np.linspace(1, 30, 30), list_vaf, 'r-')
    axs[0].set_title("VAF before BCE")
    axs[0].set_xlabel("No. of components")
    axs[0].set_ylabel("Total VAF")
    axs[0].set_ylim([0, 1])

    # Total VAF vs no. of components after BCE
    axs[1].plot(np.linspace(1, 10, 10), list_vaf_selected_cps, 'r-')
    axs[1].set_title("VAF after BCE")
    axs[1].set_xlabel("No. of components after BCE")
    #axs[1].set_ylabel("Total VAF")  
    axs[1].set_ylim([0, 1])

    # Save
    plt.savefig("fig2_VAF_of_PARAFAC.pdf", bbox_inches="tight")
    plt.savefig("fig2_VAF_of_PARAFAC.png", bbox_inches="tight")

# Plot VAF vs no. of components
VAF_combined_plots()

In [ ]:
t_rank = 11
k_ = 6

optimal = greedy_permutation(t_rank, k_, fac, tensor)
reconstruct_ = tl.cp_to_tensor((fac.weights[optimal], process_components(optimal, fac)))

# Optimal set of uncorrelated components
print("Optimal set of uncorrelated components: %s" % optimal)

factors_by_rank = process_components_by_rank(optimal, fac)

# Print VAF of each component and total VAF
sum = 0
results = []
print("\nComp\tVAF")
for i in optimal:
    print("%d\t%0.3f" % (i, result[i]))    
    sum += result[i]
    results.append(result[i])
print("\nTotal VAF with %d optimal components: %0.3f" % (len(optimal), sum))

# Generate sorted table of components
complabel = ["C" + str(n) for n in range(1,7)]
optimal_forpub = [n + 1 for n in optimal]
original_optimal_set_df = pd.DataFrame([optimal, optimal_forpub, results], index=["orig_comp", "orig_comp_forpub", "vaf"]).transpose()
sorted_optimal_set_df = original_optimal_set_df.sort_values(by="vaf", ascending=False)
sorted_optimal_set_df.insert(0, "comp_label", complabel)
sorted_optimal = sorted_optimal_set_df["orig_comp"].astype("int").to_list()
print("\nSorted optimal set of uncorrelated components: %s" % sorted_optimal)

# Print and save sorted table
print("\n")
print(sorted_optimal_set_df)
sorted_optimal_set_df.to_csv("parafac_sorted_components.txt", sep="\t")

# 6 - Find most important variables for each component
The component can be explained by its most important variables which has the highest absolute weight value.

In [ ]:
top_variables = {}
proportions = {}
prop_top_variables_names = {}
for component in range(0, 6):
    # shape 452 * 1
    ypoints_MODEA = factors_by_rank[component][0]
    # shape 30 * 1
    ypoints_MODEC = factors_by_rank[component][2]
    #shape 452 * 30
    ypoints = ypoints_MODEA.reshape(302, 1) @ ypoints_MODEC.reshape(1, 30)
    ypoints_ACLR = np.absolute(ypoints[:151][:])
    ypoints_CONTROL = np.absolute(ypoints[151:][:])

    ypoints_ACLR = ypoints_ACLR.mean(axis=0)
    ypoints_CONTROL = ypoints_CONTROL.mean(axis=0)

    data = {"ACLR": ypoints_ACLR, "CONTROL": ypoints_CONTROL}

    labels = real_variables

    df = pd.DataFrame(data, index=labels)
    #print(df)
    #print(df.ACLR.nlargest(5))
    #print(df.CONTROL.nlargest(5))
    top_variables[component] = df.ACLR.nlargest(5)
    
    # proportions should be roughly the same for each group
    score_sum = df.ACLR.sum()
    proportions[component] = 100*df.ACLR/score_sum
    prop_top_variables_names[component] = proportions[component].nlargest(5).index


# Convert proportions to df for pub, use original component numbers as column headers
proportions_df = pd.DataFrame(proportions)
prop_top_variables_names_df = pd.DataFrame(prop_top_variables_names)
proportions_df.columns = optimal   # set column headers to the original component numbers
prop_top_variables_names_df .columns = optimal

# Export unsorted tables to CSV
proportions_df.to_csv("parafac_weight_proportions.txt", sep="\t")
prop_top_variables_names_df.to_csv("parafac_top_variables.txt", sep="\t")

# Print unsorted tables
# Note: .to_string() bypasses Pandas defaults to allow stdout printing of full table
print("Weight proportions (unsorted components):")
print(proportions_df.to_string())
print("\n\nTop variables (unsorted components):")
print(prop_top_variables_names_df.to_string())


# Sort the columns by VAF and export
proportions_sorted_df = proportions_df.reindex(columns=sorted_optimal)
prop_top_variables_names_sorted_df = prop_top_variables_names_df.reindex(columns=sorted_optimal)
proportions_sorted_df.to_csv("parafac_weight_proportions_sorted.txt", sep="\t")
prop_top_variables_names_sorted_df.to_csv("parafac_top_variables_sorted.txt", sep="\t")

# Print sorted tables
# Note: .to_string() bypasses Pandas defaults to allow stdout printing of full table
print("\n\nWeight proportions (sorted components):")
print(proportions_sorted_df.to_string())
print("\n\nTop variables (sorted components):")
print(prop_top_variables_names_sorted_df.to_string())

## Quartile analysis
Plot the original data of these important variables. The graph represents the change of variable value over time. The ACLR and CONTROL group are plot respectively. By comparing the difference of line curve of two groups, we can make analysis of what landing strategy ACLR subjects commonly use with respect to these variables.

Given the component factor, variable, and original data, we can divide the loading factors for mode subject into four quartiles. Ideally, ACLR and CONTROL should be distributed into extreme quartiles(1st, 4th quartile) rather than distributed evenly. For instance, if most data points in the first quartile are ACLR, then the first quartile may refer to ACLR.

In [ ]:
def entropy(aclr_candidates, control_candidates):
    len_aclr = len(aclr_candidates)
    len_contr = len(control_candidates)
    sum_ = len_aclr + len_contr
    ratio_aclr = len_aclr / sum_
    ratio_contr = len_contr / sum_
    return -(ratio_aclr * math.log2(ratio_aclr) + ratio_contr * math.log2(ratio_contr))


def quartile(component, index_variable):
    # shape 302 * 1
    ypoints_MODEA = factors_by_rank[component][0]
    # shape 1 * 1
    ypoints_MODEC = factors_by_rank[component][2][index_variable]

    ypoints = ypoints_MODEA.reshape(302, 1) @ ypoints_MODEC.reshape(1, 1)
    ypoints = ypoints.flatten()

    ypoints_argsort = np.argsort(ypoints)
    quartile_range = [ypoints_argsort[0:75], ypoints_argsort[75:151], ypoints_argsort[151:227], ypoints_argsort[227:]]

    ret = dict()
    for index, qua in enumerate(quartile_range):
        aclr_candidates = list(filter(lambda x: x < 151, qua))
        aclr = ypoints[aclr_candidates]
        control_candidates = list(filter(lambda x: x >= 151, qua))
        control = ypoints[control_candidates]
        if len(aclr_candidates) > len(control_candidates):
            ret[index] = ["aclr", aclr_candidates, entropy(aclr_candidates, control_candidates), len(aclr_candidates), len(control_candidates)]
        else:
            ret[index] = ["control", control_candidates, entropy(aclr_candidates, control_candidates), len(aclr_candidates), len(control_candidates)]

    return ret


def find_index_by_name_variable(name):
    labels = real_variables.tolist()

    return labels.index(name)


# Example 1
c_num = 5
top_var = "lumbar_rot_moment"
index_var = find_index_by_name_variable(top_var)
ret = quartile(c_num, index_var)
print("Example 1: %s" % top_var)
for key, value in ret.items():
    print("%d\t%s\t%0.3f\t%d\t%d" % (key, value[0], value[2], value[3], value[4]))

# Example 2
c_num = 5
top_var = "lumbar_bend_moment"
index_var = find_index_by_name_variable(top_var)
ret = quartile(c_num, index_var)
print("\nExample 2: %s" % top_var)
for key, value in ret.items():
    print("%d\t%s\t%0.3f\t%d\t%d" % (key, value[0], value[2], value[3], value[4]))

## Updated variable labels for plots
Matplotlib accepts Tex-like formatting. Convert text labels to formatted text to use Greek and subscripts to minimise rework later.

In [ ]:
# Note: theta requires \\ escape otherwise it is interpreted as \t. Alternatively, use string literal r"$\theta$".

anglelabels = ["$\\theta_{HIPFLEX}$",
               "$\\theta_{HIPADD}$",
               "$\\theta_{HIPROT}$",
               "$\\theta_{KNEEEXT}$",
               "$\\theta_{KNEEROT}$",
               "$\\theta_{KNEEADD}$",
               "$\\theta_{ANKLEDF$",
               "$\\theta_{PELTILT}$",
               "$\\theta_{PELLIST}$",
               "$\\theta_{PELROT}$",
               "$\\theta_{LUMEXT}$",
               "$\\theta_{LUMBEND}$",
               "$\\theta_{LUMROT}$"]

momentlabels = ["$M_{HIPFLEX}$",
                "$M_{HIPADD}$",
                "$M_{HIPROT}$",
                "$M_{KNEEEXT}$",
                "$M_{KNEEROT}$",
                "$M_{KNEEADD}$",
                "$M_{ANKLEDF}$",
                "$M_{LUMEXT}$",
                "$M_{LUMBEND}$",
                "$M_{LUMROT}$"]

forcelabels = ["$F_{GMED}$",
               "$F_{GMAX}$",
               "$F_{RF}$",
               "$F_{VAS}$",
               "$F_{GAS}$",
               "$F_{SOL}$",
               "$F_{HAMS}$"]

# Full list
publabels = anglelabels + momentlabels + forcelabels

# Zip and convert to dict
publabeldict = dict(zip(real_variables, publabels))
    
print(pd.DataFrame.from_dict(publabeldict, orient="index", columns=["pub_label"]))

## Plot the quartiles of the original data
Given the quartiles, we can plot the original data of different quartile to observe the difference between ACLR and CONTROL over time of certain variable.

In [ ]:
def SEPARATED_GROUP_MODE_B_BY_QUARTILE(component_num, variable, index_variable, qua1, qua2, label1, label2):

    figure_B = plt.figure(dpi=300)
    
    # shape 302 * 101
    ypoints = tensor[:, :, index_variable]
    ypoints_qua1 = ypoints[qua1][:]
    ypoints_qua4 = ypoints[qua2][:]

    x = np.linspace(1, 101, 101)

    plt.plot(x, ypoints_qua1.mean(axis=0), 'r-', label=label1)
    plt.fill_between(x, ypoints_qua1.mean(axis=0) - ypoints_qua1.std(axis=0),
                     ypoints_qua1.mean(axis=0) + ypoints_qua1.std(axis=0), color='#ffa79c', alpha=0.2)

    plt.plot(x, ypoints_qua4.mean(axis=0), 'g--', label=label2)
    plt.fill_between(x, ypoints_qua4.mean(axis=0) - ypoints_qua4.std(axis=0),
                     ypoints_qua4.mean(axis=0) + ypoints_qua4.std(axis=0), color='#ceff9c', alpha=0.2)

    plt.xlabel("% landing phase")
    plt.ylabel(variable)
    plt.title("C{}: {}".format(component_num, publabeldict[variable])) # use string literal to enable Greek letters
    plt.legend()


# Plot the main variables for each component
for c in range(0, 6):
    for v in range(0, 5):

        # Component number for pub
        component_num = c + 1

        # Get the variable index
        top_variable = prop_top_variables_names_sorted_df.iloc[v, c]
        index_variable = find_index_by_name_variable(top_variable)        
        
        # Get the original unsorted component index
        component_no = sorted_optimal[c]
        unsorted_idx = optimal.index(component_no)

        # Get the quartiles
        ret = quartile(unsorted_idx, index_variable)
        qua1 = ret[3][1]
        qua2 = ret[0][1]
        label1 = ret[3][0]
        label2 = ret[0][0]

        # Plot
        SEPARATED_GROUP_MODE_B_BY_QUARTILE(component_num, top_variable, index_variable, qua1, qua2, label1, label2)


## Plots quartiles of original data for publication (split plots: C1-C3 and C4-C6)
Plot columns as components, rows as the main variables

In [ ]:
def SEPARATED_GROUP_MODE_B_BY_QUARTILE_GRID_SPLIT():
    
    # Generate separate figures for C1-C3 and C4-C6 (otherwise too crowded)
    fignum = [3, 4]
    group = ["C1-C3", "C4-C6"]
    for f in [0, 1]:

        # Variable plots for each component
        fig, axs = plt.subplots(nrows=5, ncols=3, layout="constrained", figsize=(9, 15))
        for c0 in range(0, 3):

            # Actual component index
            c = 3*f + c0
            
            for v in range(0, 5):    
    
                # Component num for pub
                component_num = c + 1

                # Get the variable index
                variable = prop_top_variables_names_sorted_df.iloc[v, c]
                index_variable = find_index_by_name_variable(variable)

                # Get the original unsorted component index
                component_no = sorted_optimal[c]
                unsorted_idx = optimal.index(component_no)

                # Get the quartiles
                ret = quartile(unsorted_idx, index_variable)
                qua1 = ret[3][1]
                qua2 = ret[0][1]
                label1 = ret[3][0]
                label2 = ret[0][0]
                
                # shape 302 * 101
                #ypoints = orig_data_tensor[:, :, index_variable]  # use original data not rescaled data
                ypoints = tensor[:, :, index_variable]  # use rescaled data
                ypoints_qua1 = ypoints[qua1][:]
                ypoints_qua4 = ypoints[qua2][:]
    
                # Time steps
                x = np.linspace(0, 100, 101)
    
                # Plot data
                ax = axs[v, c0]
                ax.plot(x, ypoints_qua1.mean(axis=0), color="orange", linestyle='-', label="Q1 (Upper)")
                ax.fill_between(x, ypoints_qua1.mean(axis=0) - ypoints_qua1.std(axis=0), ypoints_qua1.mean(axis=0) + ypoints_qua1.std(axis=0), color='orange', alpha=0.2)        
                ax.plot(x, ypoints_qua4.mean(axis=0), color="green", linestyle='--', label="Q3 (Lower)")
                ax.fill_between(x, ypoints_qua4.mean(axis=0) - ypoints_qua4.std(axis=0), ypoints_qua4.mean(axis=0) + ypoints_qua4.std(axis=0), color='green', alpha=0.2)

                
                # Indicate ACLR or Control on a nominal point on each curve
                uppery = ypoints_qua1.mean(axis=0)
                lowery = ypoints_qua4.mean(axis=0)
                # X position and offset of arrowhead
                xarrow = np.argmax(np.abs(uppery - lowery))  # place arrow at location of max difference between curves
                xoff = 10
                if (xarrow > 80):
                    xarrow = 80
                    xoff = -xoff
                elif (xarrow < 20):
                    xarrow = 20               
                # Arrow Y-offset
                if (uppery[xarrow] > lowery[xarrow]):
                    yoff = 20
                else:
                    yoff = -20
                # Labels
                if label1=="aclr":
                    lab1 = "ACLR"
                    lab2 = "Control"
                    col1 = "red"
                    col2 = "blue"
                else:
                    lab1 = "Control"
                    lab2 = "ACLR"
                    col1 = "blue"
                    col2 = "red"                
                xarrowpts = ax.transData.transform([xarrow, 0])[0]
                ax.annotate(lab1, xy=(xarrow, uppery[xarrow]), xytext=(xoff, yoff), color=col1, textcoords="offset points", arrowprops=dict(facecolor=col1, edgecolor="none", width=1, headwidth=3))
                ax.annotate(lab2, xy=(xarrow, lowery[xarrow]), xytext=(xoff, -yoff), color=col2, textcoords="offset points", arrowprops=dict(facecolor=col2, edgecolor="none", width=1, headwidth=3))

                
                # Decorations
                ax.set_xlim([0, 100])
                if v==4: ax.set_xlabel("% landing phase")
                if c0==0: ax.set_ylabel("Normalised units")                
                ax.set_title("C{}: {}".format(component_num, publabeldict[variable]))
                if (v==0) and ((c==0) or (c==3)): ax.legend(fontsize=8, frameon=False)

        # Save figures
        plt.savefig("fig{}_component_variable_quartiles_{}.pdf".format(fignum[f], group[f]), bbox_inches="tight")
        plt.savefig("fig{}_component_variable_quartiles_{}.png".format(fignum[f], group[f]), bbox_inches="tight")

# Plot the main variables for each component
SEPARATED_GROUP_MODE_B_BY_QUARTILE_GRID_SPLIT()


## Plots quartiles of original data for publication (single plot all data)
Plot columns as components, rows as the main variables

In [ ]:
def SEPARATED_GROUP_MODE_B_BY_QUARTILE_GRID_ALL():
    
    # Variable plots for each component
    fig, axs = plt.subplots(nrows=5, ncols=6, layout="constrained", figsize=(18, 15))
    for c in range(0, 6):        
        for v in range(0, 5):    

            # Component num for pub
            component_num = c + 1

            # Get the variable index
            variable = prop_top_variables_names_sorted_df.iloc[v, c]
            index_variable = find_index_by_name_variable(variable)
            
            # Get the original unsorted component index
            component_no = sorted_optimal[c]
            unsorted_idx = optimal.index(component_no)

            # Get the quartiles
            ret = quartile(c, index_variable)
            qua1 = ret[3][1]
            qua2 = ret[0][1]
            label1 = ret[3][0]
            label2 = ret[0][0]
            
            # shape 302 * 101
            #ypoints = orig_data_tensor[:, :, index_variable]  # use original data not rescaled data
            ypoints = tensor[:, :, index_variable]  # use rescaled data
            ypoints_qua1 = ypoints[qua1][:]
            ypoints_qua4 = ypoints[qua2][:]

            # Time steps
            x = np.linspace(0, 100, 101)

            # Plot data
            ax = axs[v, c]
            ax.plot(x, ypoints_qua1.mean(axis=0), color="orange", linestyle='-', label="Q1 (upper)")
            ax.fill_between(x, ypoints_qua1.mean(axis=0) - ypoints_qua1.std(axis=0), ypoints_qua1.mean(axis=0) + ypoints_qua1.std(axis=0), color='orange', alpha=0.2)        
            ax.plot(x, ypoints_qua4.mean(axis=0), color="green", linestyle='--', label="Q3 (lower)")
            ax.fill_between(x, ypoints_qua4.mean(axis=0) - ypoints_qua4.std(axis=0), ypoints_qua4.mean(axis=0) + ypoints_qua4.std(axis=0), color='green', alpha=0.2)

            # Indicate ACLR or Control on a nominal point on each curve
            uppery = ypoints_qua1.mean(axis=0)
            lowery = ypoints_qua4.mean(axis=0)
            # X position and offset of arrowhead
            xarrow = np.argmax(np.abs(uppery - lowery))  # place arrow at location of max difference between curves
            xoff = 10
            if (xarrow > 80):
                xarrow = 80
                xoff = -xoff
            elif (xarrow < 20):
                xarrow = 20               
            # Arrow Y-offset
            if (uppery[xarrow] > lowery[xarrow]):
                yoff = 20
            else:
                yoff = -20
            # Labels
            if label1=="aclr":
                lab1 = "ACLR"
                lab2 = "Control"
                col1 = "red"
                col2 = "blue"
            else:
                lab1 = "Control"
                lab2 = "ACLR"
                col1 = "blue"
                col2 = "red"                
            xarrowpts = ax.transData.transform([xarrow, 0])[0]
            ax.annotate(lab1, xy=(xarrow, uppery[xarrow]), xytext=(xoff, yoff), color=col1, textcoords="offset points", arrowprops=dict(facecolor=col1, edgecolor="none", width=1, headwidth=3))
            ax.annotate(lab2, xy=(xarrow, lowery[xarrow]), xytext=(xoff, -yoff), color=col2, textcoords="offset points", arrowprops=dict(facecolor=col2, edgecolor="none", width=1, headwidth=3))
        
            # Decorations
            ax.set_xlim([0, 100])
            if v==4: ax.set_xlabel("% landing phase")
            if c==0: ax.set_ylabel("Normalised units")
            ax.set_title("C{}: {}".format(component_num, publabeldict[variable]))
            if (v==0) and (c==0): ax.legend(frameon=False)

    # Save figures
    plt.savefig("figX_component_variable_quartiles_all.pdf", bbox_inches="tight")
    plt.savefig("figX_component_variable_quartiles_all.png", bbox_inches="tight")

# Plot the main variables for each component
SEPARATED_GROUP_MODE_B_BY_QUARTILE_GRID_ALL()


# 7 - Plot Mode A X Mode B
After knowing what this component represents, we can plot the graph of loading factors of different subjects over time to observe how the influence of these important variables changes over time for ACLR and CONTROL.

In [ ]:
def SEPARATED_GROUP_MODE_B(component):
    figure_B = plt.figure(dpi=300)
    
    # shape 302 * 1
    ypoints_MODEA = factors_by_rank[component][0]
    # shape 101 * 1
    ypoints_MODEB = factors_by_rank[component][1]
    #shape 302 * 101
    ypoints = ypoints_MODEA.reshape(302, 1) @ ypoints_MODEB.reshape(1, 101)

    ypoints_ACLR = ypoints[:151][:]
    ypoints_CONTROL = ypoints[151:][:]

    x = np.linspace(1, 101, 101)

    plt.plot(x, ypoints_ACLR.mean(axis=0), 'r-', label="ACLR")
    plt.fill_between(x, ypoints_ACLR.mean(axis=0) - ypoints_ACLR.std(axis=0),
                     ypoints_ACLR.mean(axis=0) + ypoints_ACLR.std(axis=0), color='#ffa79c', alpha=0.2)

    plt.plot(x, ypoints_CONTROL.mean(axis=0), 'g--', label="CTRL")
    plt.fill_between(x, ypoints_CONTROL.mean(axis=0) - ypoints_CONTROL.std(axis=0),
                     ypoints_CONTROL.mean(axis=0) + ypoints_CONTROL.std(axis=0), color='#ceff9c', alpha=0.2)
    plt.title("The PC coefficient waveforms of ACLR and CTRL groups for component{}".format(component+1))
    plt.legend()
    plt.xlabel("% landing phase")
    plt.ylabel("PC coefficient")


# Plot the waveforms
for i in range(6):
    
    # Get the original unsorted index of the component
    component_no = sorted_optimal[i]
    unsorted_idx = optimal.index(component_no)
    
    # Plot
    SEPARATED_GROUP_MODE_B(unsorted_idx)

## Plot mode A x mode B for publication (single plot all data)
Plot all on a single grid for publication

In [ ]:
def SEPARATED_GROUP_MODE_B_GRID_SINGLE_ROW():

    # Plots for each component
    fig, axs = plt.subplots(nrows=1, ncols=6, layout="constrained", figsize=(18, 3))    
    for c in range(0, 6):

        # Get the original unsorted index of the component
        component_no = sorted_optimal[c]
        c1 = optimal.index(component_no)        
        
        # shape 302 * 1
        ypoints_MODEA = factors_by_rank[c1][0]
        # shape 101 * 1
        ypoints_MODEB = factors_by_rank[c1][1]
        #shape 302 * 101
        ypoints = ypoints_MODEA.reshape(302, 1) @ ypoints_MODEB.reshape(1, 101)
    
        ypoints_ACLR = ypoints[:151][:]
        ypoints_CONTROL = ypoints[151:][:]

        # Time steps
        x = np.linspace(0, 100, 101)

        # Plot loadings
        ax = axs[c]
        ax.plot(x, ypoints_ACLR.mean(axis=0), color="red", linestyle='-', label="ACLR")
        ax.fill_between(x, ypoints_ACLR.mean(axis=0) - ypoints_ACLR.std(axis=0), ypoints_ACLR.mean(axis=0) + ypoints_ACLR.std(axis=0), color='red', alpha=0.2)
        ax.plot(x, ypoints_CONTROL.mean(axis=0), color="blue", linestyle='--', label="Control")
        ax.fill_between(x, ypoints_CONTROL.mean(axis=0) - ypoints_CONTROL.std(axis=0), ypoints_CONTROL.mean(axis=0) + ypoints_CONTROL.std(axis=0), color='blue', alpha=0.2)
            
        # Decorations
        ax.set_xlim([0, 100])
        ax.set_xlabel("% landing phase")
        if c==0: ax.set_ylabel("Loading")
        ax.set_title("C{}".format(str(c+1)))
        if c==0: ax.legend()

    # Save figures
    #plt.savefig("fig5_modeAxB_loadings.pdf", bbox_inches="tight") 
    #plt.savefig("fig5_modeAxB_loadings.png", bbox_inches="tight") 


def SEPARATED_GROUP_MODE_B_GRID_TWO_ROWS():

    # Plots for each component
    fig, axs0 = plt.subplots(nrows=2, ncols=3, layout="constrained", figsize=(9, 6))    
    axs = axs0.flatten()  # flatten grid from 2D indexing to 1D
    for c in range(0, 6):

        # Get the original unsorted index of the component
        component_no = sorted_optimal[c]
        c1 = optimal.index(component_no) 

        # shape 302 * 1
        ypoints_MODEA = factors_by_rank[c1][0]
        # shape 101 * 1
        ypoints_MODEB = factors_by_rank[c1][1]
        #shape 302 * 101
        ypoints = ypoints_MODEA.reshape(302, 1) @ ypoints_MODEB.reshape(1, 101)
    
        ypoints_ACLR = ypoints[:151][:]
        ypoints_CONTROL = ypoints[151:][:]

        # Time steps
        x = np.linspace(0, 100, 101)

        # Plot loadings
        ax = axs[c]
        ax.plot(x, ypoints_ACLR.mean(axis=0), color="red", linestyle='-', label="ACLR")
        ax.fill_between(x, ypoints_ACLR.mean(axis=0) - ypoints_ACLR.std(axis=0), ypoints_ACLR.mean(axis=0) + ypoints_ACLR.std(axis=0), color='red', alpha=0.2)
        ax.plot(x, ypoints_CONTROL.mean(axis=0), color="blue", linestyle='--', label="Control")
        ax.fill_between(x, ypoints_CONTROL.mean(axis=0) - ypoints_CONTROL.std(axis=0), ypoints_CONTROL.mean(axis=0) + ypoints_CONTROL.std(axis=0), color='blue', alpha=0.2)
        ax.plot(x, ypoints.mean(axis=0), color="black", label="Combined")
        
        # Decorations
        ax.set_xlim([0, 100])
        ax.set_xlabel("% landing phase")
        if c==0: ax.set_ylabel("Loading")
        ax.set_title("C{}".format(str(c+1)))
        if c==0: ax.legend(frameon=False, fontsize=8)

    # Save figures
    plt.savefig("fig5_modeAxB_loadings.pdf", bbox_inches="tight") 
    plt.savefig("fig5_modeAxB_loadings.png", bbox_inches="tight") 



# Plot loadings
SEPARATED_GROUP_MODE_B_GRID_SINGLE_ROW()
SEPARATED_GROUP_MODE_B_GRID_TWO_ROWS()        
